<a href="https://colab.research.google.com/github/dinooooooi/dinooooi/blob/main/Untitled0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install requests beautifulsoup4 pandas tqdm lxml openpyxl

In [2]:
import re
import time
import os
import requests
import pandas as pd

from bs4 import BeautifulSoup
from tqdm.auto import tqdm
from urllib.parse import quote

In [3]:
# =========================================
# Overwatch Liquipedia 설정
# =========================================

GAME_SLUG = "overwatch"
GAME_LABEL = "Overwatch 2"

BASE_URL = f"https://liquipedia.net/{GAME_SLUG}"
API_URL = f"{BASE_URL}/api.php"

# 반드시 본인 이메일/프로젝트명으로 변경
USER_AGENT = "KoreanOverwatchEsportsResearch/0.1 (your-email@example.com)"

HEADERS = {
    "User-Agent": USER_AGENT,
    "Accept-Encoding": "gzip"
}

REQUEST_SLEEP = 2.2
PARSE_SLEEP = 31

print(BASE_URL)
print(API_URL)

https://liquipedia.net/overwatch
https://liquipedia.net/overwatch/api.php


In [4]:
def clean_text(text):
    if text is None:
        return ""
    text = re.sub(r"\s+", " ", str(text))
    return text.strip()


def make_page_url(title):
    title = title or ""
    return f"{BASE_URL}/{quote(title.replace(' ', '_'))}"


def api_get(params, sleep=REQUEST_SLEEP):
    params = {
        **params,
        "format": "json"
    }

    r = requests.get(
        API_URL,
        params=params,
        headers=HEADERS,
        timeout=40
    )

    time.sleep(sleep)

    if r.status_code != 200:
        raise RuntimeError(f"HTTP {r.status_code}: {r.text[:500]}")

    data = r.json()

    if "error" in data:
        raise RuntimeError(data["error"])

    return data

In [5]:
# API 연결 테스트
data = api_get({
    "action": "query",
    "meta": "siteinfo",
    "siprop": "general"
})

data["query"]["general"]["sitename"]

'Liquipedia Overwatch Wiki'

In [6]:
def search_pages_all(keyword, limit=50, max_total=2000):
    """
    Overwatch Liquipedia Main namespace 검색 결과 전체 수집.
    페이지 본문 크롤링 X, 검색 API만 사용.
    """
    all_rows = []
    offset = 0

    while True:
        print(f"검색: {keyword} / offset={offset}")

        data = api_get({
            "action": "query",
            "list": "search",
            "srsearch": keyword,
            "srnamespace": 0,
            "srlimit": limit,
            "sroffset": offset,
        })

        results = data.get("query", {}).get("search", [])

        if not results:
            break

        for item in results:
            title = item.get("title", "")
            all_rows.append({
                "source_wiki": GAME_SLUG,
                "game_label": GAME_LABEL,
                "title": title,
                "snippet": clean_text(re.sub("<.*?>", "", item.get("snippet", ""))),
                "pageid": item.get("pageid"),
                "url": make_page_url(title),
                "search_keyword": keyword
            })

        offset += limit

        if len(all_rows) >= max_total:
            print("max_total 도달:", max_total)
            break

    return all_rows

In [7]:
OVERWATCH_SEARCH_KEYWORDS = [
    "korea",
    "South Korea",
    "Korean",
    "South Korean",
    "Korean team",
    "Korea esports",
    "Overwatch Korea",
    "Overwatch Korean",
    "Overwatch 2 Korea",
    "Overwatch 2 Korean",
    "OWCS Korea",
    "OWCS Korean",

    # 한국/관련 팀 후보
    "Cheeseburger",
    "O2 Blast",
    "Team Falcons Korea",
    "Crazy Raccoon Korea",
    "Poker Face",
    "Gen.G",
    "T1",
    "DRX",
    "Dplus KIA",
    "RunAway",
    "Element Mystic",
    "Seoul Dynasty",
    "Seoul Infernal",
    "Sin Prisa Gaming",
    "WAC",
    "Team CC Korea",
    "Old Ocean",
    "VEC",
    "PANTHERA",
    "NTMR Korea",
]

In [8]:
search_rows = []

for keyword in OVERWATCH_SEARCH_KEYWORDS:
    rows = search_pages_all(keyword, limit=50, max_total=1000)
    search_rows.extend(rows)

ow_search_df = (
    pd.DataFrame(search_rows)
    .drop_duplicates(subset=["title"])
    .reset_index(drop=True)
)

print("검색 후보 수:", len(ow_search_df))
display(ow_search_df.head(100))

검색: korea / offset=0
검색: korea / offset=50
검색: korea / offset=100
검색: korea / offset=150
검색: korea / offset=200
검색: korea / offset=250
검색: korea / offset=300
검색: korea / offset=350
검색: korea / offset=400
검색: korea / offset=450
검색: korea / offset=500
검색: korea / offset=550
검색: korea / offset=600
검색: korea / offset=650
검색: korea / offset=700
검색: korea / offset=750
검색: korea / offset=800
검색: korea / offset=850
검색: korea / offset=900
검색: korea / offset=950
max_total 도달: 1000
검색: South Korea / offset=0
검색: South Korea / offset=50
검색: South Korea / offset=100
검색: South Korea / offset=150
검색: South Korea / offset=200
검색: South Korea / offset=250
검색: South Korea / offset=300
검색: South Korea / offset=350
검색: South Korea / offset=400
검색: South Korea / offset=450
검색: South Korea / offset=500
검색: South Korea / offset=550
검색: South Korea / offset=600
검색: South Korea / offset=650
검색: South Korea / offset=700
검색: South Korea / offset=750
검색: South Korea / offset=800
검색: South Korea / offset=850
검색: S

,source_wiki,game_label,title,snippet,pageid,url,search_keyword
0,overwatch,Overwatch 2,A-Tier Tournaments,Riyadh 7 TBD TBD Overwatch Champions Series 20...,23117,https://liquipedia.net/overwatch/A-Tier_Tourna...,korea
1,overwatch,Overwatch 2,Overwatch Contenders/2022/Summer Series/Korea/...,Contenders 2022 Summer Series: Korea A-Sides L...,119189,https://liquipedia.net/overwatch/Overwatch_Con...,korea
2,overwatch,Overwatch 2,Overwatch Contenders/2022/Summer Series/Korea/...,Contenders 2022 Summer Series: Korea B-Sides L...,118630,https://liquipedia.net/overwatch/Overwatch_Con...,korea
3,overwatch,Overwatch 2,Overwatch Contenders/2023/Spring Series/Korea,Overwatch Contenders 2023 Spring Series: Korea...,122523,https://liquipedia.net/overwatch/Overwatch_Con...,korea
4,overwatch,Overwatch 2,Busan,Busan Map Information Creator: Blizzard Locati...,74940,https://liquipedia.net/overwatch/Busan,korea
...,...,...,...,...,...,...,...
95,overwatch,Overwatch 2,Overwatch Contenders/2020/Season 1/Korea/Trial...,Place Qualifies To Participant 1st OWC Week 2 ...,104401,https://liquipedia.net/overwatch/Overwatch_Con...,korea
96,overwatch,Overwatch 2,OVERWATCH Korea Cup July Online Qualifier/1,OVERWATCH Korea Cup July Online Qualifier #1 L...,98102,https://liquipedia.net/overwatch/OVERWATCH_Kor...,korea
97,overwatch,Overwatch 2,OVERWATCH Korea Cup June Online Qualifier/5,OVERWATCH Korea Cup June Online Qualifier #5 L...,94676,https://liquipedia.net/overwatch/OVERWATCH_Kor...,korea
98,overwatch,Overwatch 2,OPENER,Series 2025 - Korea Stage 2 [e] 2025-01-30 | T...,111709,https://liquipedia.net/overwatch/OPENER,korea


In [9]:
def build_ow_team_candidate_loose_df(search_df):
    team_like_keywords = [
        "team",
        "esports",
        "e-sports",
        "gaming",
        "organization",
        "overwatch",
        "owcs",

        "cheeseburger",
        "o2 blast",
        "falcons",
        "crazy raccoon",
        "poker face",
        "gen.g",
        "t1",
        "drx",
        "dplus",
        "kia",
        "runaway",
        "element mystic",
        "seoul dynasty",
        "seoul infernal",
        "sin prisa",
        "wac",
        "team cc",
        "old ocean",
        "panthera",
    ]

    bad_title_patterns = [
        "/",
        r"\b20\d{2}\b",
        "season",
        "stage",
        "major",
        "minor",
        "tournament",
        "championship",
        "qualifier",
        "results",
        "played matches",
        "matches",
        "match history",
        "statistics",
        "schedule",
        "broadcast",
        "vod",
        "interview",
        "standings",
        "bracket",
        "group stage",
        "participants",
        "rankings",
        "awards",
        "finals",
        "playoffs",
    ]

    bad_markers = [
        "player information",
        "tournament information",
        "league information",
        "match statistics",
        "broadcast talent",
    ]

    def is_candidate(row):
        title = str(row.get("title", ""))
        snippet = str(row.get("snippet", ""))
        text = f"{title} {snippet}".lower()

        if any(marker in text for marker in bad_markers):
            return False

        for pattern in bad_title_patterns:
            if re.search(pattern, title, flags=re.IGNORECASE):
                return False

        korea_terms = [
            "location: south korea",
            "region: korea",
            "south korea",
            "korea",
            "korean",
        ]

        has_korea = any(term in text for term in korea_terms)
        has_team_like = any(keyword in text for keyword in team_like_keywords)

        if "team information" in text and has_korea:
            return True

        if has_korea and has_team_like:
            return True

        return False

    df = search_df[
        search_df.apply(is_candidate, axis=1)
    ].copy().reset_index(drop=True)

    df["title_clean"] = (
        df["title"]
        .astype(str)
        .str.replace("_", " ", regex=False)
        .str.strip()
    )

    df = df.drop_duplicates(subset=["title_clean"]).reset_index(drop=True)

    print("Overwatch 완화 후보 수:", len(df))
    print("예상 parse 소요 시간:", round(len(df) * 31 / 60, 1), "분")
    print("예상 parse 소요 시간:", round(len(df) * 31 / 3600, 2), "시간")

    return df

In [10]:
ow_team_candidate_loose_df = build_ow_team_candidate_loose_df(ow_search_df)

display(ow_team_candidate_loose_df[[
    "title",
    "snippet",
    "search_keyword",
    "url"
]].head(200))

Overwatch 완화 후보 수: 288
예상 parse 소요 시간: 148.8 분
예상 parse 소요 시간: 2.48 시간


,title,snippet,search_keyword,url
0,RunAway,RunAway Team Information Location: South Korea...,korea,https://liquipedia.net/overwatch/RunAway
1,Team South Korea,South Korea announces their 2026 OWWC roster. ...,korea,https://liquipedia.net/overwatch/Team_South_Korea
2,T1,T1 Team Information Location: South Korea Regi...,korea,https://liquipedia.net/overwatch/T1
3,X6-Gaming,X6-Gaming Team Information Location: South Kor...,korea,https://liquipedia.net/overwatch/X6-Gaming
4,Lunatic-Hai,Lunatic-Hai Team Information Location: South K...,korea,https://liquipedia.net/overwatch/Lunatic-Hai
...,...,...,...,...
195,Sihu,History Upcoming Tournaments OWCS Korea Stage ...,korea,https://liquipedia.net/overwatch/Sihu
196,Ares (Korean coach),Information Name: 정승민 Romanized Name: Jeong Se...,korea,https://liquipedia.net/overwatch/Ares_%28Korea...
197,K4ne,Information Name: 최주형 Romanized Name: Choi Ju-...,korea,https://liquipedia.net/overwatch/K4ne
198,Stalk3r,Information Name: 정학용 Romanized Name: Jeong Ha...,korea,https://liquipedia.net/overwatch/Stalk3r


In [11]:
# =========================================
# Overwatch 완화 후보에서 팀 후보만 재정제
# output: ow_dedup_df2
# 페이지 불러오기 X
# =========================================

source_df = ow_team_candidate_loose_df.copy()

ow_team_name_keywords = [
    "team",
    "esports",
    "e-sports",
    "gaming",
    "organization",
    "overwatch",
    "owcs",

    "cheeseburger",
    "o2 blast",
    "falcons",
    "crazy raccoon",
    "poker face",
    "gen.g",
    "t1",
    "drx",
    "dplus",
    "kia",
    "runaway",
    "element mystic",
    "seoul dynasty",
    "seoul infernal",
    "sin prisa",
    "wac",
    "team cc",
    "old ocean",
    "panthera",
]

ow_bad_title_patterns = [
    "/",
    r"\b20\d{2}\b",
    "season",
    "stage",
    "major",
    "minor",
    "tournament",
    "championship",
    "qualifier",
    "results",
    "played matches",
    "matches",
    "match history",
    "statistics",
    "schedule",
    "broadcast",
    "vod",
    "interview",
    "standings",
    "bracket",
    "group stage",
    "participants",
    "rankings",
    "awards",
    "finals",
    "playoffs",
]

ow_bad_page_markers = [
    "player information",
    "tournament information",
    "league information",
    "match statistics",
    "broadcast talent",
]

ow_korea_terms = [
    "location: south korea",
    "region: korea",
    "south korea",
    "korea",
    "korean",
]


def is_refined_ow_team_candidate(row):
    title = str(row.get("title", ""))
    snippet = str(row.get("snippet", ""))
    text = f"{title} {snippet}".lower()

    if any(marker in text for marker in ow_bad_page_markers):
        return False

    for pattern in ow_bad_title_patterns:
        if re.search(pattern, title, flags=re.IGNORECASE):
            return False

    if not any(term in text for term in ow_korea_terms):
        return False

    if "team information" in text:
        return True

    if any(keyword in text for keyword in ow_team_name_keywords):
        return True

    return False


ow_dedup_df2 = source_df[
    source_df.apply(is_refined_ow_team_candidate, axis=1)
].copy().reset_index(drop=True)

ow_dedup_df2["title_clean"] = (
    ow_dedup_df2["title"]
    .astype(str)
    .str.replace("_", " ", regex=False)
    .str.strip()
)

ow_dedup_df2 = ow_dedup_df2.drop_duplicates(
    subset=["title_clean"]
).reset_index(drop=True)

print("Overwatch 완화 후보 수:", len(ow_team_candidate_loose_df))
print("재정제 후 ow_dedup_df2 후보 수:", len(ow_dedup_df2))
print("예상 parse 소요 시간:", round(len(ow_dedup_df2) * 31 / 60, 1), "분")
print("예상 parse 소요 시간:", round(len(ow_dedup_df2) * 31 / 3600, 2), "시간")

display(ow_dedup_df2[["title", "snippet", "search_keyword", "url"]].head(200))

Overwatch 완화 후보 수: 288
재정제 후 ow_dedup_df2 후보 수: 288
예상 parse 소요 시간: 148.8 분
예상 parse 소요 시간: 2.48 시간


,title,snippet,search_keyword,url
0,RunAway,RunAway Team Information Location: South Korea...,korea,https://liquipedia.net/overwatch/RunAway
1,Team South Korea,South Korea announces their 2026 OWWC roster. ...,korea,https://liquipedia.net/overwatch/Team_South_Korea
2,T1,T1 Team Information Location: South Korea Regi...,korea,https://liquipedia.net/overwatch/T1
3,X6-Gaming,X6-Gaming Team Information Location: South Kor...,korea,https://liquipedia.net/overwatch/X6-Gaming
4,Lunatic-Hai,Lunatic-Hai Team Information Location: South K...,korea,https://liquipedia.net/overwatch/Lunatic-Hai
...,...,...,...,...
195,Sihu,History Upcoming Tournaments OWCS Korea Stage ...,korea,https://liquipedia.net/overwatch/Sihu
196,Ares (Korean coach),Information Name: 정승민 Romanized Name: Jeong Se...,korea,https://liquipedia.net/overwatch/Ares_%28Korea...
197,K4ne,Information Name: 최주형 Romanized Name: Choi Ju-...,korea,https://liquipedia.net/overwatch/K4ne
198,Stalk3r,Information Name: 정학용 Romanized Name: Jeong Ha...,korea,https://liquipedia.net/overwatch/Stalk3r


In [12]:
def get_page_html(title):
    try:
        data = api_get({
            "action": "parse",
            "page": title,
            "prop": "text|displaytitle|categories|links"
        }, sleep=PARSE_SLEEP)

        parsed = data.get("parse", {})
        html = parsed.get("text", {}).get("*", "")
        display_title = parsed.get("displaytitle", title)

        categories = [
            c.get("*", "")
            for c in parsed.get("categories", [])
        ]

        return {
            "title": title,
            "display_title": clean_text(BeautifulSoup(display_title, "html.parser").get_text(" ", strip=True)),
            "html": html,
            "categories": categories,
            "url": make_page_url(title),
            "exists": True,
            "error": None
        }

    except Exception as e:
        return {
            "title": title,
            "display_title": title,
            "html": "",
            "categories": [],
            "url": make_page_url(title),
            "exists": False,
            "error": str(e)
        }


def extract_infobox_text(html):
    soup = BeautifulSoup(html, "lxml")

    selectors = [
        ".fo-nttax-infobox",
        ".fo-nttax-infobox-wrapper",
        ".infobox",
        "table.infobox",
    ]

    for selector in selectors:
        box = soup.select_one(selector)
        if box:
            return clean_text(box.get_text(" ", strip=True))

    return ""

In [13]:
def extract_overwatch_people_from_team_html_v4(team_title, html):
    soup = BeautifulSoup(html, "lxml")
    rows = []

    bad_link_keywords = [
        "edit", "view", "history", "team", "teams", "player", "players",
        "tournament", "tournaments", "overview", "results", "matches",
        "statistics", "schedule", "vods", "references", "news",
        "main page", "overwatch", "liquipedia",
    ]

    stop_main_sections = [
        "Results",
        "Statistics",
        "Awards",
        "Logos",
        "Media",
        "References",
        "Gallery",
        "Upcoming Matches",
        "Upcoming Tournaments",
        "Timeline",
    ]

    role_icon_map = {
        "offense": "DPS",
        "damage": "DPS",
        "dps": "DPS",
        "tank": "Tank",
        "support": "Support",
    }

    def get_prev_heading(el, levels):
        prev = el.find_previous(levels)
        if prev:
            return clean_text(prev.get_text(" ", strip=True)).replace("[edit]", "")
        return ""

    def get_main_section(el):
        return get_prev_heading(el, ["h2"])

    def get_sub_section(el):
        return get_prev_heading(el, ["h3", "h4"])

    def clean_ref_text(text):
        text = clean_text(text)
        text = re.sub(r"\[\d+\]", "", text)
        return clean_text(text)

    def extract_first_country_from_row(row_el):
        for img in row_el.select("img"):
            for attr in ["alt", "title"]:
                value = clean_text(img.get(attr))
                if not value:
                    continue
                if value.lower().endswith((".png", ".svg", ".jpg", ".webp")):
                    continue
                if value.lower() in ["edit", "view", "history"]:
                    continue
                if value.lower() in ["dps", "tank", "support", "offense"]:
                    continue
                return value
        return ""

    def extract_role_from_row(row_el):
        role_candidates = []

        # 이미지 alt/title에서 DPS/Tank/Support 추정
        for img in row_el.select("img"):
            attrs = " ".join([
                clean_text(img.get("alt")),
                clean_text(img.get("title")),
                clean_text(img.get("src")),
            ]).lower()

            for key, role in role_icon_map.items():
                if key in attrs:
                    role_candidates.append(role)

        if role_candidates:
            return role_candidates[0]

        row_text_lower = clean_text(row_el.get_text(" ", strip=True)).lower()
        for key, role in role_icon_map.items():
            if key in row_text_lower:
                return role

        return ""

    def normalize_header(h):
        h = clean_text(h).lower()

        if h in ["id", "player"]:
            return "id"
        if h == "name":
            return "name"
        if h in ["join date", "joined"]:
            return "join_date"
        if h in ["leave date", "left"]:
            return "leave_date"
        if h == "inactive date":
            return "inactive_date"
        if h in ["new team", "new_team"]:
            return "new_team"
        if h in ["country", "nationality"]:
            return "nationality"
        if h in ["role", "position"]:
            return "position"
        if h == "":
            return "position"

        return h.replace(" ", "_")

    def is_bad_person_link(name, href):
        name = clean_text(name)
        name_lower = name.lower()
        href_lower = href.lower()

        if not name:
            return True

        if name_lower in bad_link_keywords:
            return True

        if any(x.lower() in href_lower for x in [
            "/file:", "/category:", "/special:", "/help:", "/template:",
            "/portal:", "/liquipedia:", "/user:"
        ]):
            return True

        if len(name) > 80:
            return True

        if re.search(
            r"(season|stage|major|minor|playoffs|championship|cup|tournament|league|series|qualifier|finals|owcs|owl|contenders)",
            name_lower,
            flags=re.I
        ):
            return True

        return False

    def table_context(table):
        main_section = get_main_section(table)
        sub_section = get_sub_section(table)

        main_low = main_section.lower()
        sub_low = sub_section.lower()

        if any(s.lower() in main_low for s in stop_main_sections):
            return None, None, None

        if "player roster" in main_low or "former squad" in sub_low or "former players" in sub_low or "players" in sub_low:
            if "stand" in sub_low:
                return "player_roster", "Stand-in", main_section
            if "inactive" in sub_low:
                return "player_roster", "Inactive", main_section
            if "former" in sub_low:
                return "player_roster", "Former", main_section
            return "player_roster", "Active", main_section

        if "organization" in main_low or "staff" in main_low:
            if "former" in sub_low:
                return "organization", "Former", main_section
            return "organization", "Active", main_section

        return None, None, None

    def default_position(table_type, status, row_data, row_el):
        row_pos = row_data.get("position", "")
        role_from_icon = extract_role_from_row(row_el)

        if table_type == "organization":
            if row_pos:
                return row_pos
            return "Former Staff" if status == "Former" else "Staff"

        if table_type == "player_roster":
            if row_pos:
                return row_pos
            if role_from_icon:
                return role_from_icon
            if status == "Stand-in":
                return "Stand-in"
            if status == "Inactive":
                return "Inactive Player"
            if status == "Former":
                return "Former Player"
            return "Player"

        return ""

    for table_idx, table in enumerate(soup.find_all("table")):
        table_type, status, main_section = table_context(table)

        if table_type is None:
            continue

        subheading = get_sub_section(table)
        trs = table.find_all("tr")

        if not trs:
            continue

        headers = []
        header_row_idx = None

        for i, tr in enumerate(trs[:10]):
            cells = tr.find_all(["th", "td"])
            cell_texts = [clean_text(c.get_text(" ", strip=True)) for c in cells]
            lowered = [c.lower() for c in cell_texts]

            if (
                "id" in lowered
                or "player" in lowered
                or "name" in lowered
                or "join date" in lowered
                or "leave date" in lowered
                or "inactive date" in lowered
                or "organization" in lowered
            ):
                headers = cell_texts
                header_row_idx = i
                break

        if not headers:
            continue

        normalized_headers = [normalize_header(h) for h in headers]

        if len(normalized_headers) >= 4 and "position" not in normalized_headers:
            if normalized_headers[0] == "id" and normalized_headers[1] == "name":
                normalized_headers[2] = "position"

        for tr in trs[(header_row_idx + 1):]:
            cells = tr.find_all(["td", "th"])

            if not cells:
                continue

            row_text = clean_text(tr.get_text(" ", strip=True))

            if not row_text:
                continue

            if len(cells) == 1:
                continue

            row_data = {}
            local_headers = normalized_headers.copy()

            if len(cells) == len(local_headers) + 1:
                if "position" not in local_headers and "join_date" in local_headers:
                    join_idx = local_headers.index("join_date")
                    local_headers.insert(join_idx, "position")

            for idx, cell in enumerate(cells):
                if idx >= len(local_headers):
                    continue

                col = local_headers[idx]
                value = clean_text(cell.get_text(" ", strip=True))

                if col in ["join_date", "leave_date", "inactive_date"]:
                    value = clean_ref_text(value)

                row_data[col] = value

            links = []
            for a in tr.select(f'a[href^="/{GAME_SLUG}/"]'):
                link_text = clean_text(a.get_text(" ", strip=True))
                href = a.get("href", "")

                if is_bad_person_link(link_text, href):
                    continue

                links.append({
                    "name_from_link": link_text,
                    "page": href.split(f"/{GAME_SLUG}/", 1)[-1].replace("_", " "),
                    "url": "https://liquipedia.net" + href,
                })

            if links:
                main_link = links[0]
                person_id = row_data.get("id", "") or main_link["name_from_link"]
                person_page = main_link["page"]
                person_url = main_link["url"]
            else:
                person_id = row_data.get("id", "")
                person_page = ""
                person_url = ""

            if not person_id:
                continue

            if person_page.lower().replace(" ", "_") == team_title.lower().replace(" ", "_"):
                continue

            nationality = (
                row_data.get("nationality", "")
                or extract_first_country_from_row(tr)
            )

            position = default_position(table_type, status, row_data, tr)

            rows.append({
                "source_wiki": GAME_SLUG,
                "game_label": GAME_LABEL,
                "game_title": "Overwatch 2",
                "team_title": team_title,
                "team_url": make_page_url(team_title),
                "section_heading": main_section,
                "subheading": subheading,
                "table_type": table_type,
                "status": status,
                "table_index": table_idx,
                "id": person_id,
                "name": row_data.get("name", ""),
                "position": position,
                "join_date": row_data.get("join_date", ""),
                "inactive_date": row_data.get("inactive_date", ""),
                "leave_date": row_data.get("leave_date", ""),
                "new_team": row_data.get("new_team", ""),
                "nationality": nationality,
                "person_page": person_page,
                "person_url": person_url,
                "row_text": row_text,
            })

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)

    df = df.drop_duplicates(
        subset=[
            "team_title",
            "table_type",
            "status",
            "id",
            "name",
            "position",
            "join_date",
            "inactive_date",
            "leave_date",
            "person_url",
        ]
    ).reset_index(drop=True)

    return df

In [14]:
test_team = "Cheeseburger"
page = get_page_html(test_team)

print("테스트 팀:", test_team)
print("페이지 존재 여부:", page["exists"])
print("에러:", page.get("error"))

if page["exists"]:
    test_people_df = extract_overwatch_people_from_team_html_v4(
        test_team,
        page["html"]
    )

    print("추출 row 수:", len(test_people_df))

    if len(test_people_df) > 0:
        display(test_people_df[[
            "team_title",
            "game_title",
            "table_type",
            "status",
            "id",
            "name",
            "position",
            "join_date",
            "inactive_date",
            "leave_date",
            "new_team",
            "nationality",
            "person_url"
        ]].head(100))
    else:
        print("페이지는 열렸지만 row가 추출되지 않았습니다.")

테스트 팀: Cheeseburger
페이지 존재 여부: True
에러: None
추출 row 수: 17


,team_title,game_title,table_type,status,id,name,position,join_date,inactive_date,leave_date,new_team,nationality,person_url
0,Cheeseburger,Overwatch 2,player_roster,Active,Argon,Kim Han-saem,DPS,2025-08-11 [ 1 ],,,,South Korea,https://liquipedia.net/overwatch/Argon
1,Cheeseburger,Overwatch 2,player_roster,Active,FARMER,Kim Yea-han,Tank,2025-08-11 [ 1 ],,,,South Korea,https://liquipedia.net/overwatch/FARMER
2,Cheeseburger,Overwatch 2,player_roster,Active,WoochaN,Lee Woo-chan,Support,2025-08-11 [ 1 ],,,,South Korea,https://liquipedia.net/overwatch/WoochaN
3,Cheeseburger,Overwatch 2,player_roster,Active,Gur3um,Jeon Yong-min,Tank,2026-05-08 [ 14 ],,,,South Korea,https://liquipedia.net/overwatch/Gur3um
4,Cheeseburger,Overwatch 2,player_roster,Active,M1nut2,Seo Min-seo,DPS,2026-05-08 [ 14 ],,,,South Korea,https://liquipedia.net/overwatch/M1nut2
5,Cheeseburger,Overwatch 2,player_roster,Active,TenTen,,Support,2026-05-08 [ 14 ],,,,South Korea,https://liquipedia.net/overwatch/index.php?tit...
6,Cheeseburger,Overwatch 2,player_roster,Former,D0D0,Hyeon Jae-woong,DPS,2025-08-19 [ 2 ],,2025-12-18 [ 8 ],New Era,South Korea,https://liquipedia.net/overwatch/D0D0
7,Cheeseburger,Overwatch 2,player_roster,Former,SeungAn,Lee Seung-an,Tank,2025-08-11 [ 1 ],,2025-11-26 [ 5 ],INVADERS,South Korea,https://liquipedia.net/overwatch/SeungAn
8,Cheeseburger,Overwatch 2,player_roster,Former,ZeSin,Song Yun-jin,DPS,2025-08-11 [ 1 ],,2025-11-03 [ 4 ],Cheeseburger,South Korea,https://liquipedia.net/overwatch/ZeSin
9,Cheeseburger,Overwatch 2,player_roster,Former,Jamelgong,,DPS,2026-02-01 [ 9 ],,2026-05-08 [ 14 ],,South Korea,https://liquipedia.net/overwatch/index.php?tit...


In [15]:
def crawl_overwatch_teams_and_people_v4(candidate_df, output_prefix, team_type):
    team_rows = []
    people_dfs = []
    errors = []

    print("크롤링 대상 후보 수:", len(candidate_df))
    print("예상 소요 시간:", round(len(candidate_df) * 31 / 60, 1), "분")
    print("예상 소요 시간:", round(len(candidate_df) * 31 / 3600, 2), "시간")

    for title in tqdm(candidate_df["title"], desc=f"{output_prefix} 팀 검증 + 인원 추출 v4"):
        page = get_page_html(title)

        if not page["exists"]:
            team_rows.append({
                "source_wiki": GAME_SLUG,
                "game_label": GAME_LABEL,
                "title": title,
                "display_title": page["display_title"],
                "url": page["url"],
                "exists": False,
                "is_korean_team_candidate": False,
                "infobox_text": "",
                "error": page["error"],
                "team_type": team_type,
            })
            continue

        infobox = extract_infobox_text(page["html"])
        infobox_lower = infobox.lower()
        infobox_compact = infobox_lower.replace(" ", "")

        is_team_page = "team information" in infobox_lower
        is_player_page = "player information" in infobox_lower

        is_korean_team = (
            is_team_page
            and not is_player_page
            and (
                "location:southkorea" in infobox_compact
                or "region:korea" in infobox_compact
                or "location: south korea" in infobox_lower
                or "region: korea" in infobox_lower
                or "south korea" in infobox_lower
            )
        )

        team_rows.append({
            "source_wiki": GAME_SLUG,
            "game_label": GAME_LABEL,
            "title": page["title"],
            "display_title": page["display_title"],
            "url": page["url"],
            "exists": True,
            "is_korean_team_candidate": is_korean_team,
            "infobox_text": infobox,
            "error": None,
            "team_type": team_type,
        })

        if is_korean_team:
            try:
                people_df = extract_overwatch_people_from_team_html_v4(
                    page["title"],
                    page["html"]
                )

                if len(people_df) > 0:
                    people_df["team_display_title"] = page["display_title"]
                    people_df["team_url"] = page["url"]
                    people_df["team_type"] = team_type
                    people_dfs.append(people_df)

            except Exception as e:
                errors.append({
                    "team_title": page["title"],
                    "error": str(e),
                    "team_type": team_type,
                })

    teams_checked_df = pd.DataFrame(team_rows)

    teams_final_df = teams_checked_df[
        (teams_checked_df["exists"] == True)
        & (teams_checked_df["is_korean_team_candidate"] == True)
    ].copy().reset_index(drop=True)

    if people_dfs:
        people_raw_df = pd.concat(people_dfs, ignore_index=True)
    else:
        people_raw_df = pd.DataFrame()

    errors_df = pd.DataFrame(errors)

    print("검증 완료")
    print("최종 한국 팀 수:", len(teams_final_df))
    print("추출 인원 row 수:", len(people_raw_df))
    print("에러 수:", len(errors_df))

    teams_checked_path = f"/content/{output_prefix}_teams_checked.csv"
    teams_final_path = f"/content/{output_prefix}_teams_final.csv"
    people_raw_path = f"/content/{output_prefix}_people_raw.csv"
    errors_path = f"/content/{output_prefix}_errors.csv"

    teams_checked_df.to_csv(teams_checked_path, index=False, encoding="utf-8-sig")
    teams_final_df.to_csv(teams_final_path, index=False, encoding="utf-8-sig")
    people_raw_df.to_csv(people_raw_path, index=False, encoding="utf-8-sig")
    errors_df.to_csv(errors_path, index=False, encoding="utf-8-sig")

    print("저장 완료")
    print(teams_checked_path)
    print(teams_final_path)
    print(people_raw_path)
    print(errors_path)

    return teams_checked_df, teams_final_df, people_raw_df, errors_df

In [16]:
ow_teams_checked_df, ow_teams_final_df, ow_people_raw_df, ow_errors_df = crawl_overwatch_teams_and_people_v4(
    candidate_df=ow_dedup_df2,
    output_prefix="overwatch_korean_mainteam_refined_v4",
    team_type="main_team_refined"
)

display(ow_teams_final_df.head(100))
display(ow_people_raw_df.head(100))

크롤링 대상 후보 수: 288
예상 소요 시간: 148.8 분
예상 소요 시간: 2.48 시간


overwatch_korean_mainteam_refined_v4 팀 검증 + 인원 추출 v4:   0%|          | 0/288 [00:00<?, ?it/s]

검증 완료
최종 한국 팀 수: 109
추출 인원 row 수: 3042
에러 수: 0
저장 완료
/content/overwatch_korean_mainteam_refined_v4_teams_checked.csv
/content/overwatch_korean_mainteam_refined_v4_teams_final.csv
/content/overwatch_korean_mainteam_refined_v4_people_raw.csv
/content/overwatch_korean_mainteam_refined_v4_errors.csv


,source_wiki,game_label,title,display_title,url,exists,is_korean_team_candidate,infobox_text,error,team_type
0,overwatch,Overwatch 2,RunAway,RunAway,https://liquipedia.net/overwatch/RunAway,True,True,[ e ][ h ] RunAway Team Information Location: ...,None,main_team_refined
1,overwatch,Overwatch 2,Team South Korea,Team South Korea,https://liquipedia.net/overwatch/Team_South_Korea,True,True,[ e ][ h ] Team South Korea Team Information L...,None,main_team_refined
2,overwatch,Overwatch 2,T1,T1,https://liquipedia.net/overwatch/T1,True,True,[ e ][ h ] T1 Team Information Location: South...,None,main_team_refined
3,overwatch,Overwatch 2,X6-Gaming,X6-Gaming,https://liquipedia.net/overwatch/X6-Gaming,True,True,[ e ][ h ] X6-Gaming Team Information Location...,None,main_team_refined
4,overwatch,Overwatch 2,Lunatic-Hai,Lunatic-Hai,https://liquipedia.net/overwatch/Lunatic-Hai,True,True,[ e ][ h ] Lunatic-Hai Team Information Locati...,None,main_team_refined
...,...,...,...,...,...,...,...,...,...,...
95,overwatch,Overwatch 2,Nc Wolves,Nc Wolves,https://liquipedia.net/overwatch/Nc_Wolves,True,True,[ e ][ h ] Nc Wolves Team Information Location...,None,main_team_refined
96,overwatch,Overwatch 2,EXL-Esports,EXL-Esports,https://liquipedia.net/overwatch/EXL-Esports,True,True,[ e ][ h ] EXL-Esports Team Information Locati...,None,main_team_refined
97,overwatch,Overwatch 2,GameHome Monsters,GameHome Monsters,https://liquipedia.net/overwatch/GameHome_Mons...,True,True,[ e ][ h ] GameHome Monsters Team Information ...,None,main_team_refined
98,overwatch,Overwatch 2,Mighty Storm,Mighty Storm,https://liquipedia.net/overwatch/Mighty_Storm,True,True,[ e ][ h ] Mighty Storm Team Information Locat...,None,main_team_refined


,source_wiki,game_label,game_title,team_title,team_url,section_heading,subheading,table_type,status,table_index,...,join_date,inactive_date,leave_date,new_team,nationality,person_page,person_url,row_text,team_display_title,team_type
0,overwatch,Overwatch 2,Overwatch 2,RunAway,https://liquipedia.net/overwatch/RunAway,Player Roster,Former,player_roster,Former,0,...,2016-??-??,,2016-??-??,,South Korea,index.php?title=Mono (Korean player)&action=ed...,https://liquipedia.net/overwatch/index.php?tit...,Mono Kim Hyung-sun Tank 2016-??-?? 2016-??-??,RunAway,main_team_refined
1,overwatch,Overwatch 2,Overwatch 2,RunAway,https://liquipedia.net/overwatch/RunAway,Player Roster,Former,player_roster,Former,0,...,2016-??-??,,2016-??-??,,South Korea,index.php?title=Quad (Korean player)&action=ed...,https://liquipedia.net/overwatch/index.php?tit...,Quad DPS 2016-??-?? 2016-??-??,RunAway,main_team_refined
2,overwatch,Overwatch 2,Overwatch 2,RunAway,https://liquipedia.net/overwatch/RunAway,Player Roster,Former,player_roster,Former,1,...,2017-08-03 [ 7 ],,2017-12-11 [ 9 ],OpTic Academy,South Korea,Kaiser,https://liquipedia.net/overwatch/Kaiser,Kaiser Ryu Sang-hoon Tank 2017-08-03 [ 7 ] 201...,RunAway,main_team_refined
3,overwatch,Overwatch 2,Overwatch 2,RunAway,https://liquipedia.net/overwatch/RunAway,Player Roster,Former,player_roster,Former,1,...,2017-07-07 [ 5 ],,2017-08-07 [ 8 ],RunAway (General Manager),South Korea,Runner,https://liquipedia.net/overwatch/Runner,Runner Yoon Dae-hoon Support (Inactive) 2017-0...,RunAway,main_team_refined
4,overwatch,Overwatch 2,Overwatch 2,RunAway,https://liquipedia.net/overwatch/RunAway,Player Roster,Former,player_roster,Former,1,...,2017-04-24,,2017-07-07 [ 6 ],,South Korea,Mirage,https://liquipedia.net/overwatch/Mirage,Mirage Bae Jung-min Flex 2017-04-24 2017-07-07...,RunAway,main_team_refined
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,overwatch,Overwatch 2,Overwatch 2,T1,https://liquipedia.net/overwatch/T1,Organization,Active,organization,Active,7,...,2025-01-24 [ 33 ],,,,South Korea,RUSH,https://liquipedia.net/overwatch/RUSH,RUSH Yun Hee-won Head Coach (Head Coach) 2025-...,T1,main_team_refined
96,overwatch,Overwatch 2,Overwatch 2,T1,https://liquipedia.net/overwatch/T1,Organization,Active,organization,Active,7,...,2025-04-30 [ 36 ],,,,United States,Avast,https://liquipedia.net/overwatch/Avast,Avast Connor Prince Content Creator (Content C...,T1,main_team_refined
97,overwatch,Overwatch 2,Overwatch 2,T1,https://liquipedia.net/overwatch/T1,Organization,Active,organization,Active,7,...,2025-06-21 [ 37 ],,,,South Korea,index.php?title=Ichi (Korean content creator)&...,https://liquipedia.net/overwatch/index.php?tit...,Ichi Content Creator (Influencer) 2025-06-21 [...,T1,main_team_refined
98,overwatch,Overwatch 2,Overwatch 2,T1,https://liquipedia.net/overwatch/T1,Organization,Active,organization,Active,7,...,2026-02-06 [ 42 ],,,,France,Poko,https://liquipedia.net/overwatch/Poko,Poko Gael Gouzerch Content Creator (Content Cr...,T1,main_team_refined


In [17]:
def make_unique_people_latest_from_raw(people_raw_df):
    """
    raw 데이터에서 사람 단위 unique 매핑.

    기준:
    1. person_url이 있으면 person_url 기준으로 같은 사람 묶음
    2. person_url이 없으면 id + name 기준으로 묶음
    3. join_date가 가장 최신인 row 우선
    4. join_date가 없으면 inactive_date, 그다음 leave_date를 대체 날짜로 사용
    5. 같은 날짜면 Stand-in > Inactive > Active > Former 우선
    6. 사람당 1행만 남김
    """

    if people_raw_df is None or len(people_raw_df) == 0:
        return pd.DataFrame()

    df = people_raw_df.copy()

    required_cols = [
        "person_url",
        "id",
        "name",
        "position",
        "status",
        "join_date",
        "inactive_date",
        "leave_date",
        "team_title",
        "team_url",
        "nationality",
        "new_team",
    ]

    for col in required_cols:
        if col not in df.columns:
            df[col] = ""

    df["person_unique_key"] = (
        df["person_url"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    empty_url_mask = df["person_unique_key"] == ""

    df.loc[empty_url_mask, "person_unique_key"] = (
        df.loc[empty_url_mask, "id"].fillna("").astype(str).str.lower().str.strip()
        + "|"
        + df.loc[empty_url_mask, "name"].fillna("").astype(str).str.lower().str.strip()
    )

    df = df[
        df["person_unique_key"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("|")
    ].copy()

    def extract_date(value):
        value = "" if pd.isna(value) else str(value)
        matched = re.search(r"(\d{4}-\d{2}-\d{2})", value)
        if matched:
            return matched.group(1)
        return ""

    df["join_date_clean"] = df["join_date"].apply(extract_date)
    df["inactive_date_clean"] = df["inactive_date"].apply(extract_date)
    df["leave_date_clean"] = df["leave_date"].apply(extract_date)

    df["join_date_parsed"] = pd.to_datetime(df["join_date_clean"], errors="coerce")
    df["inactive_date_parsed"] = pd.to_datetime(df["inactive_date_clean"], errors="coerce")
    df["leave_date_parsed"] = pd.to_datetime(df["leave_date_clean"], errors="coerce")

    df["effective_date_parsed"] = (
        df["join_date_parsed"]
        .fillna(df["inactive_date_parsed"])
        .fillna(df["leave_date_parsed"])
    )

    status_priority_map = {
        "Stand-in": 5,
        "Inactive": 4,
        "Active": 3,
        "Former": 2,
        "": 1,
    }

    df["status_priority"] = (
        df["status"]
        .fillna("")
        .astype(str)
        .str.strip()
        .map(status_priority_map)
        .fillna(1)
        .astype(int)
    )

    position_lower = df["position"].fillna("").astype(str).str.lower()

    df.loc[position_lower.str.contains("stand", na=False), "status_priority"] = df[
        "status_priority"
    ].clip(lower=5)

    df.loc[position_lower.str.contains("inactive", na=False), "status_priority"] = df[
        "status_priority"
    ].clip(lower=4)

    df = df.sort_values(
        by=[
            "person_unique_key",
            "effective_date_parsed",
            "join_date_parsed",
            "status_priority",
            "inactive_date_parsed",
            "leave_date_parsed",
            "team_title",
        ],
        ascending=[
            True,
            False,
            False,
            False,
            False,
            False,
            True,
        ],
        na_position="last"
    )

    unique_df = df.drop_duplicates(
        subset=["person_unique_key"],
        keep="first"
    ).reset_index(drop=True)

    unique_df = unique_df.drop(
        columns=[
            "join_date_clean",
            "inactive_date_clean",
            "leave_date_clean",
            "join_date_parsed",
            "inactive_date_parsed",
            "leave_date_parsed",
            "effective_date_parsed",
            "status_priority",
        ],
        errors="ignore"
    )

    return unique_df

In [18]:
ow_people_unique_df = make_unique_people_latest_from_raw(ow_people_raw_df)

print("raw row 수:", len(ow_people_raw_df))
print("최신 기준 unique 인원 수:", len(ow_people_unique_df))

display(ow_people_unique_df[[
    "game_title",
    "team_title",
    "status",
    "id",
    "name",
    "position",
    "join_date",
    "inactive_date",
    "leave_date",
    "new_team",
    "nationality",
    "person_url"
]].head(100))

raw row 수: 3042
최신 기준 unique 인원 수: 1329


,game_title,team_title,status,id,name,position,join_date,inactive_date,leave_date,new_team,nationality,person_url
0,Overwatch 2,EXL-RAISO,Former,Captain,Park Won-Jae (박원재),DPS,2018-02-23,,2018-03-07 [ 1 ],,South Korea,
1,Overwatch 2,Fnatic,Active,cArn,Patrik Sättermon,CGO,2012-04-??,,,,Sweden,
2,Overwatch 2,Team Falcons,Former,7sN,Hassan Aljurais,Manager (Manager),2021-04-04 [ 2 ],,2022-04-06 [ 7 ],,Saudi Arabia,https://liquipedia.net/overwatch/7sN
3,Overwatch 2,ZAN Esports,Former,A1IEN,Han Yu-bin,DPS,2026-02-15 [ 2 ],,2026-05-15 [ 8 ],,South Korea,https://liquipedia.net/overwatch/A1IEN
4,Overwatch 2,ONSIDE GAMING,Active,A1M,An Yoon-su,Staff (Owner),2025-03-19 [ 2 ],,,,South Korea,https://liquipedia.net/overwatch/A1M
...,...,...,...,...,...,...,...,...,...,...,...,...
95,Overwatch 2,X6-Gaming,Former,BeBe,Yoon Hee-chang,Support,2017-02-??,,2018-09-??,Hangzhou Spark,South Korea,https://liquipedia.net/overwatch/BeBe
96,Overwatch 2,WLGaming,Former,BePo,Kim Min-jae (김민재),Support,2018-10-07 [ 11 ],,2018-10-??,,South Korea,https://liquipedia.net/overwatch/BePo
97,Overwatch 2,Seoul Dynasty,Former,beast,Kwangjin Baek,General Manager/Head Coach,2017-08-22,,2018-07-16 [ 14 ],,South Korea,https://liquipedia.net/overwatch/Beast_(Kwang-...
98,Overwatch 2,ZAN Esports,Former,Becky,Kim Il-ha,DPS,2026-02-15 [ 2 ],,2026-05-03 [ 5 ],ZANSIDE GAMING,South Korea,https://liquipedia.net/overwatch/Becky


In [19]:
ow_dedup_df2.to_csv(
    "/content/overwatch_team_candidates_refined.csv",
    index=False,
    encoding="utf-8-sig"
)

ow_teams_checked_df.to_csv(
    "/content/overwatch_teams_checked_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

ow_teams_final_df.to_csv(
    "/content/overwatch_teams_final_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

ow_people_raw_df.to_csv(
    "/content/overwatch_people_raw_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

ow_people_unique_df.to_csv(
    "/content/overwatch_people_unique_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

ow_errors_df.to_csv(
    "/content/overwatch_extract_errors_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료")
print("/content/overwatch_teams_final_v4.csv")
print("/content/overwatch_people_unique_v4.csv")

저장 완료
/content/overwatch_teams_final_v4.csv
/content/overwatch_people_unique_v4.csv
